# AI Smart Water Monitoring — Exploratory Data Analysis

This notebook provides interactive EDA for the training dataset.

**Run the training pipeline first** to ensure the dataset is available:
```
python src/pipeline/training_pipeline.py
```

---
Sections:
1. Load & inspect the dataset
2. Missing value analysis
3. Class distribution
4. Sensor distributions
5. Sensor boxplots by class
6. Correlation heatmap
7. Turbidity / water quality
8. Time-series view (if createdAt exists)
9. Water-loss target (if available)

In [ ]:
# Setup — run from the project root or add it to sys.path
import sys
from pathlib import Path

# If running from notebooks/, go up one level to project root
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config.config import TRAINING_FILE, TURBIDITY_COLUMN, TIMESTAMP_COLUMN, WATER_LOSS_TARGET
from src.data.excel_loader import load_training_data

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print('Project root:', PROJECT_ROOT)
print('Training file:', TRAINING_FILE)

## 1. Load & Inspect the Dataset

In [ ]:
df = load_training_data()
print(f'\nShape: {df.shape}')
df.head(10)

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].style.background_gradient(cmap='Oranges')

## 3. Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['predictionData'].value_counts().sort_index()
bars = ax.bar(['NORMAL (0)', 'LEAK (1)'], counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val), ha='center', fontsize=12)
ax.set_title('Class Distribution (predictionData)')
ax.set_ylabel('Record Count')
plt.tight_layout()
plt.show()
print(counts)

## 4. Sensor Distributions

In [ ]:
sensor_cols = ['flowSensorData', 'pressureSensorData', 'tankLevelSensorData']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, sensor_cols):
    df[col].hist(ax=ax, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(f'{col}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('Sensor Value Distributions', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Sensor Boxplots by Leak Class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, sensor_cols):
    df.boxplot(column=col, by='predictionData', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('predictionData (0=NORMAL, 1=LEAK)')
plt.suptitle('Sensor Readings by Leak Class')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 7. Turbidity / Water Quality

In [ ]:
if TURBIDITY_COLUMN in df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    df[TURBIDITY_COLUMN].hist(bins=40, ax=ax, color='darkorange', edgecolor='white')
    ax.axvline(30, color='green', linestyle='--', linewidth=2, label='Good/Moderate (30)')
    ax.axvline(60, color='red',   linestyle='--', linewidth=2, label='Moderate/Poor (60)')
    ax.set_title('Turbidity Distribution')
    ax.set_xlabel('Turbidity Value')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Water quality breakdown
    def quality_label(v):
        if v <= 30: return 'GOOD'
        elif v <= 60: return 'MODERATE'
        else: return 'POOR'

    df['_wq'] = df[TURBIDITY_COLUMN].apply(quality_label)
    print(df['_wq'].value_counts())
    df.drop(columns=['_wq'], inplace=True)
else:
    print(f'Column "{TURBIDITY_COLUMN}" not found in dataset.')

## 8. Time-Series View (if createdAt exists)

In [ ]:
if TIMESTAMP_COLUMN in df.columns:
    df_ts = df.copy()
    df_ts[TIMESTAMP_COLUMN] = pd.to_datetime(df_ts[TIMESTAMP_COLUMN], errors='coerce')
    df_ts = df_ts.dropna(subset=[TIMESTAMP_COLUMN]).sort_values(TIMESTAMP_COLUMN)

    fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
    for ax, col in zip(axes, sensor_cols):
        ax.plot(df_ts[TIMESTAMP_COLUMN], df_ts[col], linewidth=0.8, color='steelblue')
        # Highlight leak events
        leak_mask = df_ts['predictionData'] == 1
        ax.scatter(df_ts.loc[leak_mask, TIMESTAMP_COLUMN], df_ts.loc[leak_mask, col],
                   color='red', s=15, label='LEAK', zorder=5)
        ax.set_ylabel(col)
        ax.legend(loc='upper right')
    axes[-1].set_xlabel('Time')
    plt.suptitle('Sensor Readings Over Time (red = LEAK)', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{TIMESTAMP_COLUMN}" not found — time-series plot skipped.')

## 9. Water-Loss Target (if available)

In [ ]:
if WATER_LOSS_TARGET in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    df[WATER_LOSS_TARGET].hist(bins=40, ax=axes[0], color='purple', edgecolor='white')
    axes[0].set_title(f'Distribution: {WATER_LOSS_TARGET}')
    axes[0].set_xlabel('Water Loss Rate')

    axes[1].scatter(df['flowSensorData'], df[WATER_LOSS_TARGET], alpha=0.4, color='purple', s=15)
    axes[1].set_xlabel('flowSensorData')
    axes[1].set_ylabel(WATER_LOSS_TARGET)
    axes[1].set_title('Flow vs Water Loss Rate')

    plt.tight_layout()
    plt.show()
    print(df[WATER_LOSS_TARGET].describe())
else:
    print(f'Column "{WATER_LOSS_TARGET}" not found — water-loss EDA skipped.')
    print('This is expected if your dataset does not include water-loss measurements.')